# 🌪️ NLP with Disaster Tweets — RoBERTa 5-Fold Ensemble

> **Public Score: 0.84125** | Top ~5% on the Leaderboard

**Author:** Hasan Pireci ([@codelones](https://github.com/codelones))  
**GitHub:** [nlp-disaster-tweets](https://github.com/codelones)

---

### 🎯 Project Summary

This notebook walks through a complete NLP pipeline for classifying disaster tweets.
Starting from simple TF-IDF baselines, we progressively improve to a **RoBERTa 5-Fold Ensemble**
achieving a public score of **0.84125**.

| Approach | Val F1 | Public Score |
|----------|--------|-------------|
| TF-IDF + Naive Bayes | 0.6035 | — |
| TF-IDF + LinearSVC | 0.5537 | — |
| DistilBERT (cleaned text) | 0.7787 | — |
| RoBERTa (raw text) | 0.8262 | 0.83266 |
| **RoBERTa 5-Fold Ensemble** | **~0.81 avg** | **0.84125** |

---

### 💡 Key Insights

1. **Raw text > cleaned text for BERT** — hashtags like `#wildfire` carry disaster signals
2. **RoBERTa > DistilBERT** — better pre-training leads to better fine-tuning
3. **5-Fold Ensemble > Single model** — averaging probabilities reduces variance
4. **Submission format matters** — always verify target distribution before submitting

---

### 📋 Pipeline

```
Raw Data → EDA → Text Cleaning (baseline only) → TF-IDF Baseline
→ DistilBERT Fine-tuning → RoBERTa Fine-tuning → 5-Fold CV Ensemble → Submission
```


## 1. Imports & Setup

In [ ]:
!pip install transformers -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import html
import torch
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_score, StratifiedKFold
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW
from transformers import (DistilBertTokenizer, DistilBertForSequenceClassification,
                          RobertaTokenizer, RobertaForSequenceClassification,
                          get_linear_schedule_with_warmup)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Data Loading

The dataset contains tweets labeled as:
- **1** → Real disaster tweet
- **0** → Not a disaster tweet


In [ ]:
train = pd.read_csv('/kaggle/input/competitions/nlp-getting-started/train.csv')
test = pd.read_csv('/kaggle/input/competitions/nlp-getting-started/test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape : {test.shape}")
print(f"\nColumns: {train.columns.tolist()}")
print(f"\nMissing values:")
print(train.isnull().sum())

## 3. Exploratory Data Analysis

### 3.1 Target Distribution


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Target distribution
train['target'].value_counts().plot(kind='bar', ax=axes[0],
    color=['steelblue', 'darkorange'], edgecolor='white')
axes[0].set_title('Target Distribution', fontsize=12, fontweight='bold')
axes[0].set_xticklabels(['Not Disaster (0)', 'Disaster (1)'], rotation=0)
axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}\n({p.get_height()/len(train)*100:.1f}%)',
                    (p.get_x() + p.get_width()/2., p.get_height()),
                    ha='center', va='bottom', fontsize=10)

# Tweet length by class
train['text_length'] = train['text'].apply(len)
train[train['target']==1]['text_length'].plot(kind='hist', bins=40, alpha=0.7,
    ax=axes[1], label='Disaster', color='red')
train[train['target']==0]['text_length'].plot(kind='hist', bins=40, alpha=0.7,
    ax=axes[1], label='Not Disaster', color='steelblue')
axes[1].set_title('Tweet Length by Class', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Character Count')
axes[1].legend()

# Word count by class
train['word_count'] = train['text'].apply(lambda x: len(x.split()))
train[train['target']==1]['word_count'].plot(kind='hist', bins=30, alpha=0.7,
    ax=axes[2], label='Disaster', color='red')
train[train['target']==0]['word_count'].plot(kind='hist', bins=30, alpha=0.7,
    ax=axes[2], label='Not Disaster', color='steelblue')
axes[2].set_title('Word Count by Class', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Word Count')
axes[2].legend()

plt.tight_layout()
plt.show()

print("\nAverage stats by class:")
print(train.groupby('target')[['text_length', 'word_count']].mean().round(2))

### 3.2 Sample Tweets

In [ ]:
pd.set_option('display.max_colwidth', 120)
print("=== DISASTER TWEETS (target=1) ===")
print(train[train['target']==1]['text'].sample(5, random_state=42).to_string())
print("\n=== NOT DISASTER TWEETS (target=0) ===")
print(train[train['target']==0]['text'].sample(5, random_state=42).to_string())

## 4. Text Cleaning (for Baseline Models)

> **Note:** We use cleaned text for TF-IDF baseline models only.
> For BERT/RoBERTa, we use **raw text** — the model already understands context,
> and hashtags like `#wildfire` carry important disaster signals we don't want to remove.

### What we clean:
- **HTML entities**: `&amp;` → `&`
- **URLs**: `http://...` removed
- **Mentions**: `@username` removed  
- **Hashtags**: `#disaster` removed
- **Special characters**: Only letters and spaces kept
- **Stopwords**: Common words like "the", "a", "is" removed


In [ ]:
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = html.unescape(text)                      # &amp; → &
    text = re.sub(r'http\S+', '', text)             # Remove URLs
    text = re.sub(r'@\w+', '', text)                # Remove mentions
    text = re.sub(r'#\w+', '', text)                # Remove hashtags
    text = re.sub(r'[^a-zA-Z\s]', '', text)         # Keep letters only
    text = text.lower()                              # Lowercase
    text = re.sub(r'\s+', ' ', text).strip()        # Clean whitespace
    text = ' '.join([w for w in text.split()         # Remove stopwords
                     if w not in stop_words])
    return text

train['clean_text'] = train['text'].apply(clean_text)
test['clean_text'] = test['text'].apply(clean_text)

# Show before/after
print("BEFORE:", train['text'].iloc[0])
print("AFTER :", train['clean_text'].iloc[0])

## 5. TF-IDF Baseline Models

### What is TF-IDF?
- **TF (Term Frequency)**: How often does a word appear in this tweet?
- **IDF (Inverse Document Frequency)**: How rare is this word across all tweets?
- Rare but frequent words get high scores → better signal

### Limitation
TF-IDF treats each word independently — it cannot understand context.
"fire" in "building on fire" and "fire an employee" get the same representation.
This is why we need BERT later.


In [ ]:
# TF-IDF with unigrams and bigrams
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(train['clean_text'])
X_test_tfidf = tfidf.transform(test['clean_text'])

print(f"Feature matrix shape: {X_train_tfidf.shape}")
print(f"Each tweet is now a vector of {X_train_tfidf.shape[1]} features")

In [ ]:
# Compare multiple models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0),
    'LinearSVC'          : LinearSVC(max_iter=1000),
    'Naive Bayes'        : MultinomialNB(),
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train_tfidf, train['target'],
                             cv=5, scoring='f1')
    results[name] = scores.mean()
    print(f"{name:<25} F1: {scores.mean():.4f} ± {scores.std():.4f}")

# Visualize
fig, ax = plt.subplots(figsize=(8, 4))
names = list(results.keys())
scores = list(results.values())
colors = ['#2E75B6' if s == max(scores) else '#AED6F1' for s in scores]
bars = ax.bar(names, scores, color=colors, edgecolor='white')
ax.set_ylabel('F1 Score')
ax.set_title('TF-IDF Baseline Comparison', fontsize=12, fontweight='bold')
ax.set_ylim(0.5, 0.7)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
            f'{score:.4f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print(f"\nBest baseline: Naive Bayes with F1={max(scores):.4f}")
print("\nLimitation: TF-IDF cannot capture context.")
print("'fire' in 'building on fire' = 'fire' in 'fire an employee'")
print("→ We need BERT to understand context!")

## 6. Helper Functions for BERT Training


In [ ]:
class TweetDataset(Dataset):
    """PyTorch Dataset for tweet tokenizations."""
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def train_epoch(model, loader, optimizer, scheduler, device):
    """Train for one epoch, return average loss."""
    model.train()
    total_loss = 0
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids=input_ids,
                       attention_mask=attention_mask,
                       labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def eval_epoch(model, loader, device):
    """Evaluate model, return F1 score."""
    model.eval()
    preds, true_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions = torch.argmax(outputs.logits, dim=1)
            preds.extend(predictions.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    return f1_score(true_labels, preds)


print("Helper functions defined.")

## 7. DistilBERT Baseline

### What is BERT?
BERT (Bidirectional Encoder Representations from Transformers) is a pre-trained
deep learning model that understands **context**:

- "building on **fire**" → disaster context
- "**firing** an employee" → non-disaster context

Same word, different meaning — BERT handles this perfectly.

### Why DistilBERT?
DistilBERT is a lighter version of BERT:
- **40% smaller** and **60% faster** than BERT
- Retains **97%** of BERT's performance
- Great for quick experimentation

### Fine-tuning vs Training from Scratch
We don't train BERT from scratch — it would require billions of sentences and months of compute.
Instead, we **fine-tune** it: we take a model pre-trained on massive text data
and adapt the last classification layers to our specific task.


In [ ]:
# Tokenization
tokenizer_distil = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# We use CLEANED text for DistilBERT (to show the difference with RoBERTa later)
train_enc_distil = tokenizer_distil(
    list(train['clean_text']),
    max_length=128, truncation=True,
    padding='max_length', return_tensors='pt'
)
test_enc_distil = tokenizer_distil(
    list(test['clean_text']),
    max_length=128, truncation=True,
    padding='max_length', return_tensors='pt'
)

print(f"Train shape: {train_enc_distil['input_ids'].shape}")
print(f"\nExample tokenization:")
sample = "building on fire near downtown"
tokens = tokenizer_distil(sample)
print(f"Text   : {sample}")
print(f"Tokens : {tokenizer_distil.convert_ids_to_tokens(tokens['input_ids'])}")
print(f"IDs    : {tokens['input_ids']}")

In [ ]:
# Build datasets
train_ds_distil = TweetDataset(train_enc_distil, list(train['target']))
test_ds_distil = TweetDataset(test_enc_distil)

# Train/val split
val_size = int(0.1 * len(train_ds_distil))
train_size = len(train_ds_distil) - val_size
train_sub, val_sub = random_split(train_ds_distil, [train_size, val_size],
                                   generator=torch.Generator().manual_seed(42))

train_loader_distil = DataLoader(train_sub, batch_size=32, shuffle=True)
val_loader_distil = DataLoader(val_sub, batch_size=32, shuffle=False)
test_loader_distil = DataLoader(test_ds_distil, batch_size=32, shuffle=False)

# Load model
model_distil = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2)
model_distil = model_distil.to(device)

# Optimizer and scheduler
epochs = 3
optimizer_distil = AdamW(model_distil.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader_distil) * epochs
scheduler_distil = get_linear_schedule_with_warmup(
    optimizer_distil,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

# Training
print("Training DistilBERT...")
best_distil_f1 = 0
distil_history = []

for epoch in range(epochs):
    loss = train_epoch(model_distil, train_loader_distil,
                      optimizer_distil, scheduler_distil, device)
    val_f1 = eval_epoch(model_distil, val_loader_distil, device)
    distil_history.append({'epoch': epoch+1, 'loss': loss, 'val_f1': val_f1})
    print(f"Epoch {epoch+1}/{epochs} | Loss: {loss:.4f} | Val F1: {val_f1:.4f}")
    if val_f1 > best_distil_f1:
        best_distil_f1 = val_f1
        torch.save(model_distil.state_dict(), 'best_distilbert.pt')
        print(f"  ✓ Saved! Best F1: {best_distil_f1:.4f}")

print(f"\nDistilBERT Best Val F1: {best_distil_f1:.4f}")

## 8. RoBERTa — Better Pre-training, Better Results

### Why RoBERTa over BERT/DistilBERT?

RoBERTa (Robustly Optimized BERT Pretraining Approach) was trained:
- On **10x more data** than BERT
- With **longer training** and larger batch sizes
- Without Next Sentence Prediction (which BERT uses but isn't always helpful)

### Key Change: Raw Text Instead of Cleaned Text

For DistilBERT we used cleaned text. Here we switch to **raw text**.

Why? Because:
- `#wildfire` → hashtag signals disaster
- `@FEMA` → mention of emergency agency signals disaster  
- `http://emergency.gov` → URL context matters

BERT-based models are smart enough to ignore noise while preserving signal.
Cleaning text removes valuable information.


In [ ]:
# RoBERTa tokenizer - raw text this time!
tokenizer_roberta = RobertaTokenizer.from_pretrained('roberta-base')

train_enc_roberta = tokenizer_roberta(
    list(train['text']),   # RAW TEXT - no cleaning
    max_length=160, truncation=True,
    padding='max_length', return_tensors='pt'
)
test_enc_roberta = tokenizer_roberta(
    list(test['text']),    # RAW TEXT
    max_length=160, truncation=True,
    padding='max_length', return_tensors='pt'
)

# Datasets
train_ds_roberta = TweetDataset(train_enc_roberta, list(train['target']))
test_ds_roberta = TweetDataset(test_enc_roberta)

val_size = int(0.1 * len(train_ds_roberta))
train_size = len(train_ds_roberta) - val_size
train_sub_r, val_sub_r = random_split(train_ds_roberta, [train_size, val_size],
                                       generator=torch.Generator().manual_seed(42))

train_loader_roberta = DataLoader(train_sub_r, batch_size=32, shuffle=True)
val_loader_roberta = DataLoader(val_sub_r, batch_size=32, shuffle=False)
test_loader_roberta = DataLoader(test_ds_roberta, batch_size=32, shuffle=False)

# Model
model_roberta = RobertaForSequenceClassification.from_pretrained(
    'roberta-base', num_labels=2)
model_roberta = model_roberta.to(device)

# Train
epochs = 3
optimizer_roberta = AdamW(model_roberta.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader_roberta) * epochs
scheduler_roberta = get_linear_schedule_with_warmup(
    optimizer_roberta,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

print("Training RoBERTa (raw text)...")
best_roberta_f1 = 0
roberta_history = []

for epoch in range(epochs):
    loss = train_epoch(model_roberta, train_loader_roberta,
                      optimizer_roberta, scheduler_roberta, device)
    val_f1 = eval_epoch(model_roberta, val_loader_roberta, device)
    roberta_history.append({'epoch': epoch+1, 'loss': loss, 'val_f1': val_f1})
    print(f"Epoch {epoch+1}/{epochs} | Loss: {loss:.4f} | Val F1: {val_f1:.4f}")
    if val_f1 > best_roberta_f1:
        best_roberta_f1 = val_f1
        torch.save(model_roberta.state_dict(), 'best_roberta.pt')
        print(f"  ✓ Saved! Best F1: {best_roberta_f1:.4f}")

print(f"\nRoBERTa Best Val F1: {best_roberta_f1:.4f}")
print(f"Improvement over DistilBERT: +{best_roberta_f1 - best_distil_f1:.4f}")

### Model Progress Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training curves
epochs_list = [h['epoch'] for h in distil_history]
distil_f1s = [h['val_f1'] for h in distil_history]
roberta_f1s = [h['val_f1'] for h in roberta_history]

axes[0].plot(epochs_list, distil_f1s, 'o-', label='DistilBERT', color='steelblue')
axes[0].plot(epochs_list, roberta_f1s, 's-', label='RoBERTa', color='darkorange')
axes[0].set_title('Val F1 by Epoch', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('F1 Score')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Overall comparison
models_comp = ['TF-IDF\nNaive Bayes', 'TF-IDF\nLinearSVC',
               'DistilBERT\n(clean)', 'RoBERTa\n(raw)']
scores_comp = [0.6035, 0.5537, best_distil_f1, best_roberta_f1]
colors_comp = ['#AED6F1', '#AED6F1', '#2E75B6', '#E74C3C']

bars = axes[1].bar(models_comp, scores_comp, color=colors_comp, edgecolor='white')
axes[1].set_title('Model Comparison', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Val F1 Score')
axes[1].set_ylim(0.4, 0.9)
for bar, score in zip(bars, scores_comp):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{score:.4f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 9. 5-Fold Cross-Validation Ensemble

### Why Ensemble?

A single model trained on one train/val split might be lucky or unlucky
depending on which samples end up in validation.

**5-Fold CV Ensemble:**
1. Split data into 5 equal folds
2. Train a separate model on each fold (4 folds train, 1 fold val)
3. Each model predicts on test set as **probabilities**
4. Average the 5 probability predictions
5. Take argmax for final class

This reduces variance and typically improves public score by **0.005-0.015**.

### Why Probabilities Instead of Classes?

Averaging probabilities `[0.3, 0.7]` is more informative than averaging classes `[0, 1]`.
Soft voting captures model confidence.


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
test_preds_all = np.zeros((len(test), 2))
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train, train['target'])):
    print(f"\n{'='*50}")
    print(f"FOLD {fold+1}/5")
    print(f"{'='*50}")

    train_fold = train.iloc[train_idx]
    val_fold = train.iloc[val_idx]

    # Tokenize
    train_fold_enc = tokenizer_roberta(
        list(train_fold['text']), max_length=160,
        truncation=True, padding='max_length', return_tensors='pt')
    val_fold_enc = tokenizer_roberta(
        list(val_fold['text']), max_length=160,
        truncation=True, padding='max_length', return_tensors='pt')

    # Datasets
    train_fold_ds = TweetDataset(train_fold_enc, list(train_fold['target']))
    val_fold_ds = TweetDataset(val_fold_enc, list(val_fold['target']))
    train_fold_loader = DataLoader(train_fold_ds, batch_size=32, shuffle=True)
    val_fold_loader = DataLoader(val_fold_ds, batch_size=32, shuffle=False)

    # Model
    fold_model = RobertaForSequenceClassification.from_pretrained(
        'roberta-base', num_labels=2)
    fold_model = fold_model.to(device)

    fold_optimizer = AdamW(fold_model.parameters(), lr=2e-5, weight_decay=0.01)
    total_steps = len(train_fold_loader) * 3
    fold_scheduler = get_linear_schedule_with_warmup(
        fold_optimizer,
        num_warmup_steps=total_steps // 10,
        num_training_steps=total_steps)

    best_fold_f1 = 0
    for epoch in range(3):
        loss = train_epoch(fold_model, train_fold_loader,
                          fold_optimizer, fold_scheduler, device)
        val_f1 = eval_epoch(fold_model, val_fold_loader, device)
        print(f"  Epoch {epoch+1}/3 | Loss: {loss:.4f} | Val F1: {val_f1:.4f}")
        if val_f1 > best_fold_f1:
            best_fold_f1 = val_f1
            torch.save(fold_model.state_dict(), f'fold_{fold+1}.pt')

    fold_scores.append(best_fold_f1)
    print(f"  Fold {fold+1} Best F1: {best_fold_f1:.4f}")

    # Predict probabilities on test
    fold_model.load_state_dict(torch.load(f'fold_{fold+1}.pt'))
    fold_model.eval()
    fold_preds = []

    with torch.no_grad():
        for batch in test_loader_roberta:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = fold_model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
            fold_preds.append(probs)

    fold_preds = np.vstack(fold_preds)
    test_preds_all += fold_preds

print(f"\n{'='*50}")
print(f"CV Results: {[f'{s:.4f}' for s in fold_scores]}")
print(f"Mean F1: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

## 10. Generate Final Submission

In [ ]:
# Average probabilities from 5 folds
final_preds = np.argmax(test_preds_all, axis=1)

submission = pd.DataFrame({'id': test['id'], 'target': final_preds})

print("Prediction distribution:")
print(submission['target'].value_counts())
print(f"\nRatio: {submission['target'].mean():.3f} positive")

submission.to_csv('submission.csv', index=False)
print("\nsubmission.csv ready!")

In [ ]:
# Visualize fold results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fold scores
axes[0].bar([f'Fold {i+1}' for i in range(5)], fold_scores,
            color=['#2E75B6']*5, edgecolor='white')
axes[0].axhline(y=np.mean(fold_scores), color='red', linestyle='--',
                label=f'Mean: {np.mean(fold_scores):.4f}')
axes[0].set_title('F1 Score per Fold', fontsize=12, fontweight='bold')
axes[0].set_ylabel('F1 Score')
axes[0].set_ylim(0.7, 0.9)
axes[0].legend()
for i, score in enumerate(fold_scores):
    axes[0].text(i, score + 0.003, f'{score:.4f}', ha='center', fontsize=10)

# Final comparison
final_comparison = {
    'TF-IDF\nNaive Bayes': 0.6035,
    'DistilBERT': best_distil_f1,
    'RoBERTa\nSingle': best_roberta_f1,
    'RoBERTa\n5-Fold': np.mean(fold_scores)
}
colors_final = ['#AED6F1', '#2E75B6', '#E67E22', '#E74C3C']
bars = axes[1].bar(list(final_comparison.keys()),
                   list(final_comparison.values()),
                   color=colors_final, edgecolor='white')
axes[1].set_title('Overall Model Progression', fontsize=12, fontweight='bold')
axes[1].set_ylabel('F1 Score')
axes[1].set_ylim(0.4, 0.9)
for bar, score in zip(bars, final_comparison.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{score:.4f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 11. Key Learnings & Takeaways

### What We Learned

| Concept | Insight |
|---------|---------|
| **TF-IDF** | Fast baseline but ignores context — polysemy and synonymy are unsolvable |
| **BERT Fine-tuning** | Transfer learning: use pre-trained knowledge for new tasks |
| **Raw vs Cleaned Text** | For BERT models, raw text preserves more signal |
| **RoBERTa > DistilBERT** | Better pre-training → better fine-tuning results |
| **5-Fold Ensemble** | Averaging probabilities reduces variance, improves generalization |
| **Submission Verification** | Always check target distribution before submitting |

---

### Score Progression

| Version | Approach | Public Score |
|---------|----------|-------------|
| v1 | TF-IDF + Naive Bayes | ~0.60 |
| v2 | DistilBERT (cleaned text) | ~0.78 |
| v3 | RoBERTa (raw text) | 0.83266 |
| **v4** | **RoBERTa 5-Fold Ensemble** | **0.84125** |

---

### If I Were to Continue...
- Try `roberta-large` (more parameters, higher ceiling)
- Add `keyword` feature as additional context
- Try DeBERTa — currently state-of-the-art for NLP classification
- Pseudo-labeling: use high-confidence test predictions as additional training data

---

*If this notebook was helpful, please upvote! ⬆️*

**Author:** Hasan Pireci | [GitHub](https://github.com/codelones) | hasanpireci92@gmail.com
